# ModelForge Lite — Phase 6: Evaluation Harness

This notebook pulls all four variants' results and scores them on three axes:
- **Relevance** — does the answer actually address the question?
- **Faithfulness** — is the answer grounded / not hallucinated?
- **Latency** — how fast was each variant (already recorded in earlier phases)

We use an LLM-as-judge approach for relevance and faithfulness: a separate model call scores each answer on a simple rubric. This is a standard, explainable technique — not a black box.

In [ ]:
!pip install -q transformers accelerate huggingface_hub pandas

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Pull all four results files and merge them

Since all four notebooks ran the same `eval.csv` in the same row order, we can merge on row index.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

HF_USERNAME = "YOUR_HF_USERNAME"
dataset_repo = f"{HF_USERNAME}/modelforge-lite-support-data"

files = {
    "baseline": "results/baseline_results.csv",
    "rag": "results/rag_results.csv",
    "finetuned": "results/finetuned_results.csv",
    "finetuned_rag": "results/finetuned_rag_results.csv",
}

dfs = {}
for key, path in files.items():
    local_path = hf_hub_download(repo_id=dataset_repo, filename=path, repo_type="dataset")
    dfs[key] = pd.read_csv(local_path)
    print(f"{key}: {len(dfs[key])} rows")

In [ ]:
merged = dfs["baseline"][["question", "intent", "base_model_answer", "base_model_latency_sec"]].copy()
merged["rag_answer"] = dfs["rag"]["rag_answer"]
merged["rag_latency_sec"] = dfs["rag"]["rag_latency_sec"]
merged["finetuned_answer"] = dfs["finetuned"]["finetuned_answer"]
merged["finetuned_latency_sec"] = dfs["finetuned"]["finetuned_latency_sec"]
merged["finetuned_rag_answer"] = dfs["finetuned_rag"]["finetuned_rag_answer"]
merged["finetuned_rag_latency_sec"] = dfs["finetuned_rag"]["finetuned_rag_latency_sec"]

print(f"Merged: {len(merged)} rows, {len(merged.columns)} columns")
merged.head()

## 2. Load a judge model

We use a small instruct model as the judge — separate from the models being evaluated, so it's not scoring its own outputs. Qwen2.5-1.5B-Instruct is still light enough for a free T4 but has enough reasoning ability to give sensible scores.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

JUDGE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("Judge model loaded.")

## 3. Write the scoring function

The judge is given the question and the answer, and asked to output two 1-5 scores in a strict, parseable format. Keeping the rubric simple and the output format rigid makes this reliable enough for a small project like this.

In [ ]:
import re

JUDGE_PROMPT_TEMPLATE = """You are evaluating a customer support AI's answer.

Question: {question}
Answer: {answer}

Score the answer on two dimensions, from 1 (worst) to 5 (best):
1. RELEVANCE: Does the answer directly address the question asked?
2. FAITHFULNESS: Does the answer sound like a plausible, grounded support response, without inventing suspicious specific claims (like exact dates, dollar amounts, or policy numbers) that weren't asked for?

Respond in EXACTLY this format, nothing else:
RELEVANCE: <score>
FAITHFULNESS: <score>"""

def judge_answer(question, answer):
    prompt_text = JUDGE_PROMPT_TEMPLATE.format(question=question, answer=answer)
    messages = [{"role": "user", "content": prompt_text}]
    prompt = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = judge_tokenizer(prompt, return_tensors="pt").to(judge_model.device)

    with torch.no_grad():
        output = judge_model.generate(**inputs, max_new_tokens=30, do_sample=False)

    generated = output[0][inputs["input_ids"].shape[1]:]
    text = judge_tokenizer.decode(generated, skip_special_tokens=True)

    relevance = re.search(r"RELEVANCE:\s*(\d)", text)
    faithfulness = re.search(r"FAITHFULNESS:\s*(\d)", text)

    return (
        int(relevance.group(1)) if relevance else None,
        int(faithfulness.group(1)) if faithfulness else None,
    )

# sanity check
print(judge_answer("How can I get a refund?", "You can request a refund from your account's order page within 30 days."))

## 4. Score every answer, for all four variants

This means 4 variants × 40 questions = 160 judge calls. On a free T4 with this small judge model, expect roughly 10-15 minutes.

In [ ]:
variants = {
    "base_model": "base_model_answer",
    "rag": "rag_answer",
    "finetuned": "finetuned_answer",
    "finetuned_rag": "finetuned_rag_answer",
}

for variant_key, answer_col in variants.items():
    relevances, faithfulnesses = [], []
    for i, row in merged.iterrows():
        rel, faith = judge_answer(row["question"], row[answer_col])
        relevances.append(rel)
        faithfulnesses.append(faith)
        if (i + 1) % 10 == 0:
            print(f"{variant_key}: [{i+1}/{len(merged)}] scored")
    merged[f"{variant_key}_relevance"] = relevances
    merged[f"{variant_key}_faithfulness"] = faithfulnesses

print("All variants scored.")

## 5. Build the summary comparison table

This is THE table for the call — one row per variant, showing average relevance, average faithfulness, and average latency.

In [ ]:
latency_cols = {
    "base_model": "base_model_latency_sec",
    "rag": "rag_latency_sec",
    "finetuned": "finetuned_latency_sec",
    "finetuned_rag": "finetuned_rag_latency_sec",
}

summary_rows = []
for variant_key in variants.keys():
    summary_rows.append({
        "variant": variant_key,
        "avg_relevance": round(merged[f"{variant_key}_relevance"].mean(), 2),
        "avg_faithfulness": round(merged[f"{variant_key}_faithfulness"].mean(), 2),
        "avg_latency_sec": round(merged[latency_cols[variant_key]].mean(), 2),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## 6. Push results to Hugging Face

In [ ]:
from huggingface_hub import HfApi

merged.to_csv("full_eval_scored.csv", index=False)
summary_df.to_csv("eval_summary.csv", index=False)

api = HfApi()
for fname in ["full_eval_scored.csv", "eval_summary.csv"]:
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=f"results/{fname}",
        repo_id=dataset_repo,
        repo_type="dataset",
    )
print("Uploaded full_eval_scored.csv and eval_summary.csv")

## Done — Phase 6 checklist

- [ ] All four variants' results merged into one table
- [ ] LLM-as-judge scored every answer on relevance + faithfulness
- [ ] Summary table built: one row per variant, avg relevance / faithfulness / latency
- [ ] Both `full_eval_scored.csv` and `eval_summary.csv` pushed to Hugging Face

## What to be able to explain on the call
- **Why LLM-as-judge instead of a fixed formula?** Relevance and faithfulness are subjective/semantic qualities — hard to capture with exact string matching. Using a separate model as a judge, with a strict rubric and output format, is a standard, widely-used technique for this kind of evaluation, while still being explainable (you can show the exact prompt and rubric used).
- **Why a different, separate model as judge?** Using the same model to judge its own answers risks it favoring its own style/quirks. A separate model gives a more independent signal.
- **What does the summary table actually show?** Walk through it row by row: base model likely lowest on faithfulness/specificity, RAG improves grounding, fine-tuned improves tone/relevance, fine-tuned+RAG likely best on both — but check your actual numbers, since real results can surprise you, and that's fine to discuss honestly.
- **Limitations worth naming proactively:** small eval set (40 questions), a small judge model rather than a large frontier model, single-run scores with no repeated sampling. Naming these yourself, unprompted, is a strong signal of engineering maturity — it shows you understand the difference between "a demo" and "a rigorous benchmark," and that you know how you'd extend this into the latter.

Next: Phase 7 — the FastAPI backend and Streamlit demo app that ties everything together for the live call.